# Modbus 프로토콜 학습 및 서버 실습

---

## 1. Modbus 란?

**Modbus**는 1979년 Modicon(현 Schneider Electric)이 개발한 **산업용 직렬 통신 프로토콜**입니다.  
PLC, 센서, 인버터 등 공장 자동화 장비 간 데이터 교환에 가장 널리 사용됩니다.

### 통신 방식
| 방식 | 설명 |
|------|------|
| **Modbus RTU** | RS-232/RS-485 직렬 통신 (바이너리) |
| **Modbus ASCII** | RS-232/RS-485 직렬 통신 (ASCII 문자) |
| **Modbus TCP** | 이더넷 TCP/IP 기반 (표준 포트: **502**) |

> 이번 실습은 **Modbus TCP** 방식, **포트 5020** 사용

---

## 2. 클라이언트 / 디바이스 구조

> Modbus 공식 사양(2012년 이후)은 Master/Slave 대신 **Client/Server** 또는 **Client/Device** 용어를 사용합니다.  
> pymodbus 3.x도 이에 맞게 API 명칭을 변경했습니다 (`ModbusSlaveContext` → `ModbusDeviceContext`).

```
┌─────────────┐   요청(Request)    ┌─────────────┐
│  클라이언트  │ ─────────────────► │  디바이스    │
│  (Client)   │ ◄───────────────── │  (Device)   │
└─────────────┘   응답(Response)   └─────────────┘
```

- **클라이언트(Client)**: 통신을 주도, 읽기/쓰기 요청 송신 → *클라이언트 노트북*
- **디바이스(Device)**: 요청에 응답, 데이터 보유 → *이 서버 노트북*
- 디바이스는 최대 **247개** 까지 연결 가능 (Unit ID 1~247)

---

## 3. Modbus 데이터 영역 (4가지)

| 영역 이름 | 주소 접두어 | 데이터 타입 | 클라이언트 접근 | 실제 용도 |
|-----------|------------|------------|----------------|----------|
| **Coil** | 0x (0xxxx) | 비트 (0/1) | 읽기 + **쓰기** | 디지털 출력 (릴레이, 램프) |
| **Discrete Input** | 1x (1xxxx) | 비트 (0/1) | 읽기 전용 | 디지털 입력 (버튼, 센서) |
| **Input Register** | 3x (3xxxx) | 워드 (16bit) | 읽기 전용 | 아날로그 입력 (온도, 압력) |
| **Holding Register** | 4x (4xxxx) | 워드 (16bit) | 읽기 + **쓰기** | 설정값 저장 (속도, 목표값) |

### Function Code (FC) 표

| FC | 기능 | 대상 영역 |
|----|------|-----------|
| 01 | Read Coils | Coil |
| 02 | Read Discrete Inputs | Discrete Input |
| 03 | Read Holding Registers | Holding Register |
| 04 | Read Input Registers | Input Register |
| 05 | Write Single Coil | Coil |
| 06 | Write Single Register | Holding Register |
| 15 | Write Multiple Coils | Coil |
| 16 | Write Multiple Registers | Holding Register |

---

## 4. Modbus TCP 패킷 구조 (MBAP Header)

```
┌──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│Transaction│ Protocol │  Length  │  Unit ID │  FC Code │   Data   │
│  ID (2B)  │  ID (2B) │   (2B)   │   (1B)   │   (1B)   │  (nB)   │
└──────────┴──────────┴──────────┴──────────┴──────────┴──────────┘
```

- **Transaction ID**: 요청/응답 매칭용 일련번호
- **Protocol ID**: 항상 0x0000 (Modbus)
- **Unit ID**: 디바이스 주소 (이번 실습: **1**)

---
## 실습 환경 구성

```
docker-compose.yml
│
├── server-lab (이 노트북)
│     ├── JupyterLab  → http://localhost:8889  (token: py)
│     └── Modbus 서버 → 0.0.0.0:5020
│
└── client-lab (클라이언트 노트북)
      └── JupyterLab  → http://localhost:8888  (token: py)
            └── server-lab:5020 으로 접속
```

### 실습 순서
1. **이 노트북**: 아래 셀을 순서대로 실행 → Modbus 서버 기동 (Step 3까지)
2. **클라이언트 노트북(8888)**: 서버에 연결하여 읽기/쓰기 실습
3. 서버에서 값을 변경(Step 5, 7)하면서 클라이언트에서 실시간 확인

---
## [Step 1] 라이브러리 임포트 및 버전 확인

In [20]:
import asyncio
import pymodbus
from pymodbus.server import StartAsyncTcpServer
from pymodbus.datastore import ModbusSequentialDataBlock, ModbusDeviceContext, ModbusServerContext

print(f"pymodbus 버전: {pymodbus.__version__}")
print("임포트 완료")

pymodbus 버전: 3.13.1
임포트 완료


---
## [Step 2] 데이터모델 구성

서버가 보유할 데이터 영역을 초기화합니다.

| 영역 | 변수 | 주소 | 초기값 |
|------|------|------|---------|
| Coil (비트 읽/쓰기) | `co_block` | 0 ~ 19 | 모두 False |
| Discrete Input (비트 읽기전용) | `di_block` | 0 ~ 19 | 모두 False |
| Holding Register (워드 읽/쓰기) | `hr_block` | 0 ~ 19 | 모두 0 |
| Input Register (워드 읽기전용) | `ir_block` | 0 ~ 19 | 모두 0 |

> `ModbusSequentialDataBlock(시작주소, [초기값 리스트])`

In [1]:
from pymodbus.datastore import ModbusSequentialDataBlock, ModbusDeviceContext, ModbusServerContext

co_block = ModbusSequentialDataBlock(0, [False] * 20)
di_block = ModbusSequentialDataBlock(0, [False] * 20)
hr_block = ModbusSequentialDataBlock(0, [0] * 20)
ir_block = ModbusSequentialDataBlock(0, [0] * 20)

store = ModbusDeviceContext(co=co_block, di=di_block, hr=hr_block, ir=ir_block)
context = ModbusServerContext(devices=store, single=True)

print("데이터스토어 초기화 완료")
print(f"  Coil (fc=1) 초기값: {context[0x00].getValues(1, 0, count=10)}")
print(f"  HR   (fc=3) 초기값: {context[0x00].getValues(3, 0, count=10)}")


데이터스토어 초기화 완료
  Coil (fc=1) 초기값: [False, False, False, False, False, False, False, False, False, False]
  HR   (fc=3) 초기값: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


---
## [Step 3] Modbus TCP 서버 시작 (포트 5020)

Jupyter는 이미 이벤트 루프가 실행 중이므로 `asyncio.create_task()`로 백그라운드 실행합니다.

> ⚠️ `asyncio.run()`은 사용 불가 — 이미 실행 중인 루프와 충돌합니다.

In [8]:
import asyncio
# 💡 DataType 없이 원천 차단하는 구버전/신버전 호환 코드
from pymodbus.datastore import ModbusSequentialDataBlock, ModbusDeviceContext, ModbusServerContext
from pymodbus.server import StartAsyncTcpServer

# 1 기반 주소로 세팅하여 주소 오류 예방
co_block = ModbusSequentialDataBlock(1, [False] * 20)   # Coil
di_block = ModbusSequentialDataBlock(1, [False] * 20)   # Discrete Input
hr_block = ModbusSequentialDataBlock(1, [0] * 20)       # Holding Register
ir_block = ModbusSequentialDataBlock(1, [0] * 20)       # Input Register

store = ModbusDeviceContext(co=co_block, di=di_block, hr=hr_block, ir=ir_block)
context = ModbusServerContext(devices=store, single=True)
print("데이터스토어(context) 초기화 완료!")

# 3. 백그라운드로 서버 실행 (포트 15020)
server_task = asyncio.create_task(
    StartAsyncTcpServer(context, address=("0.0.0.0", 15020))
)

await asyncio.sleep(1)

print("========================================")
print(" Modbus TCP 서버 실행 중")
print(" 주소: 0.0.0.0:15020")
print("========================================")

데이터스토어(context) 초기화 완료!
 Modbus TCP 서버 실행 중
 주소: 0.0.0.0:15020


---
## [Step 4] 코일(비트) 현재 값 확인

서버 내부에서 직접 데이터스토어를 조회합니다.  
`context[0x00].getValues(fc, address, count)`
- `fc=1` → Coil 영역
- `address` → 시작 주소
- `count` → 읽을 개수

In [9]:
coil_values = context[0x00].getValues(1, 0, count=10)
print("[현재 코일 값] 주소 0~9")
print("-" * 30)
for i, v in enumerate(coil_values):
    status = "ON " if v else "OFF"
    bar = "■" if v else "□"
    print(f"  코일[{i}] = {bar} {status}")

[현재 코일 값] 주소 0~9
------------------------------
  코일[0] = □ OFF
  코일[1] = □ OFF
  코일[2] = □ OFF
  코일[3] = □ OFF
  코일[4] = □ OFF
  코일[5] = □ OFF
  코일[6] = □ OFF
  코일[7] = □ OFF
  코일[8] = □ OFF
  코일[9] = □ OFF


In [10]:
# fc=1: Coil 영역에 값 쓰기 (주소 0부터 10개)
new_coils = [True, False, True, True, False, False, True, False, True, False]
context[0x00].setValues(1, 0, new_coils)

# 변경 확인
coil_values = context[0x00].getValues(1, 0, count=10)
print("[변경 후 코일 값] 주소 0~9")
print("-" * 30)
for i, v in enumerate(coil_values):
    status = "ON " if v else "OFF"
    bar = "■" if v else "□"
    print(f"  코일[{i}] = {bar} {status}")
print()
print("★ 클라이언트 노트북에서 코일 읽기 셀을 다시 실행해 보세요!")

[변경 후 코일 값] 주소 0~9
------------------------------
  코일[0] = ■ ON 
  코일[1] = □ OFF
  코일[2] = ■ ON 
  코일[3] = ■ ON 
  코일[4] = □ OFF
  코일[5] = □ OFF
  코일[6] = ■ ON 
  코일[7] = □ OFF
  코일[8] = ■ ON 
  코일[9] = □ OFF

★ 클라이언트 노트북에서 코일 읽기 셀을 다시 실행해 보세요!


---
## [Step 6] 홀딩 레지스터(워드) 현재 값 확인

- `fc=3` → Holding Register 영역
- 각 레지스터는 **16비트 정수** (0 ~ 65535)
- 실수 표현 시 × 0.1 또는 × 0.01 배율 사용 (예: 250 → 25.0℃)

In [11]:
reg_values = context[0x00].getValues(3, 0, count=10)
print("[현재 레지스터 값] 주소 0~9")
print("-" * 30)
for i, v in enumerate(reg_values):
    print(f"  레지스터[{i}] = {v}")

[현재 레지스터 값] 주소 0~9
------------------------------
  레지스터[0] = 0
  레지스터[1] = 0
  레지스터[2] = 0
  레지스터[3] = 0
  레지스터[4] = 0
  레지스터[5] = 0
  레지스터[6] = 0
  레지스터[7] = 0
  레지스터[8] = 0
  레지스터[9] = 0


---
## [Step 7] 홀딩 레지스터(워드) 값 변경

산업 현장에서 온도, 압력, 속도 등의 측정값을 레지스터에 저장하는 상황을 시뮬레이션합니다.  
이 셀 실행 후 **클라이언트 노트북에서 레지스터 읽기를 다시 실행**해 보세요.

In [12]:
# fc=3: Holding Register 영역에 값 쓰기 (주소 0부터 10개)
# 예시: 온도(×0.1℃), 기압(hPa), 속도(rpm), 기타
new_regs = [250, 1013, 1500, 400, 500, 1000, 2000, 3000, 4000, 5000]
context[0x00].setValues(3, 0, new_regs)

# 변경 확인
reg_values = context[0x00].getValues(3, 0, count=10)
labels = ["온도(×0.1℃)", "기압(hPa) ", "속도(rpm) ",
          "주소3      ", "주소4      ", "주소5      ",
          "주소6      ", "주소7      ", "주소8      ", "주소9      "]
print("[변경 후 레지스터 값] 주소 0~9")
print("-" * 40)
for i, (v, label) in enumerate(zip(reg_values, labels)):
    print(f"  레지스터[{i}] = {v:5d}  ({label})")
print()
print("★ 클라이언트 노트북에서 레지스터 읽기 셀을 다시 실행해 보세요!")

[변경 후 레지스터 값] 주소 0~9
----------------------------------------
  레지스터[0] =   250  (온도(×0.1℃))
  레지스터[1] =  1013  (기압(hPa) )
  레지스터[2] =  1500  (속도(rpm) )
  레지스터[3] =   400  (주소3      )
  레지스터[4] =   500  (주소4      )
  레지스터[5] =  1000  (주소5      )
  레지스터[6] =  2000  (주소6      )
  레지스터[7] =  3000  (주소7      )
  레지스터[8] =  4000  (주소8      )
  레지스터[9] =  5000  (주소9      )

★ 클라이언트 노트북에서 레지스터 읽기 셀을 다시 실행해 보세요!


---
## [Step 8] 서버 중지

실습이 끝나면 아래 셀로 서버를 중지합니다.

In [13]:
server_task.cancel()
try:
    await server_task
except asyncio.CancelledError:
    pass
print("Modbus 서버 중지 완료")

Modbus 서버 중지 완료
